In [4]:
import torch
import torch.nn as nn

In [5]:
class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # 输入到隐藏状态的权重
        self.W_x = nn.Linear(input_size, 4 * hidden_size)
        # 上一时刻隐藏状态到当前隐藏状态的权重
        self.W_h = nn.Linear(hidden_size, 4 * hidden_size)
        
    def forward(self, x, hidden_state):
        # 解包 hidden_state，包含 (h_t-1, C_t-1)
        h_prev, c_prev = hidden_state

        # 计算门控机制：包含遗忘门、输入门、候选记忆单元和输出门
        gates = self.W_x(x) + self.W_h(h_prev)
        
        # 分割成四个部分：分别为遗忘门、输入门、候选记忆单元和输出门
        f_gate, i_gate, candidate, o_gate = torch.chunk(gates, 4, dim=1)

        # 遗忘门：用 sigmoid 激活，控制遗忘多少前一时刻的信息
        f_gate = torch.sigmoid(f_gate)

        # 输入门：用 sigmoid 激活，控制新信息的写入
        i_gate = torch.sigmoid(i_gate)

        # 候选记忆单元：用 tanh 激活，生成新候选信息
        candidate = torch.tanh(candidate)

        # 输出门：用 sigmoid 激活，控制哪些信息传递到下一个隐藏状态
        o_gate = torch.sigmoid(o_gate)

        # 更新记忆单元：前一时刻的记忆单元通过遗忘门保留部分，当前时刻的候选记忆单元通过输入门写入
        c_t = f_gate * c_prev + i_gate * candidate

        # 更新隐藏状态：当前记忆单元通过 tanh 激活后，通过输出门决定传递哪些信息
        h_t = o_gate * torch.tanh(c_t)

        # 返回新的隐藏状态 (h_t, c_t)
        return h_t, (h_t, c_t)
    

# 测试自定义的LSTMCell
input_size = 3   # 输入维度
hidden_size = 5  # 隐藏状态维度

# 实例化LSTMCell
lstm_cell = LSTMCell(input_size, hidden_size)

# 创建一个时间步的输入数据和初始状态
x = torch.randn(1, input_size)        # 输入数据，维度为 (batch_size, input_size)
h_prev = torch.zeros(1, hidden_size)  # 上一个时间步的隐藏状态
c_prev = torch.zeros(1, hidden_size)  # 上一个时间步的记忆单元状态

# 执行前向传播
h_t, (h_next, c_next) = lstm_cell(x, (h_prev, c_prev))

print("输出的隐藏状态 h_t:", h_t)
print("下一个时间步的隐藏状态 h_next:", h_next)
print("下一个时间步的记忆单元状态 c_next:", c_next)

输出的隐藏状态 h_t: tensor([[-0.1445, -0.0851,  0.0344,  0.0640,  0.2264]], grad_fn=<MulBackward0>)
下一个时间步的隐藏状态 h_next: tensor([[-0.1445, -0.0851,  0.0344,  0.0640,  0.2264]], grad_fn=<MulBackward0>)
下一个时间步的记忆单元状态 c_next: tensor([[-0.4355, -0.1604,  0.2644,  0.1370,  0.4991]], grad_fn=<AddBackward0>)


In [7]:
import torch
import torch.nn as nn

# 定义LSTM模型
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM层
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        # 全连接层，用于输出预测结果
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # 初始化隐藏状态和记忆单元状态
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # 隐藏状态
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # 记忆单元状态

        # LSTM 前向传播
        out, _ = self.lstm(x, (h0, c0))
        
        # 取最后一个时间步的输出
        out = self.fc(out[:, -1, :])  # 取最后一个时间步的隐藏状态进行预测
        return out

# 测试LSTM模型
input_size = 10    # 输入维度
hidden_size = 20   # 隐藏层维度
num_layers = 2     # LSTM层数
output_size = 1    # 输出维度

# 实例化模型
model = LSTMModel(input_size, hidden_size, num_layers, output_size)

# 创建随机输入数据 (batch_size, seq_length, input_size)
x = torch.randn(5, 7, input_size)  # 例如 batch_size=5, 序列长度=7, 输入维度=10
output = model(x)
print(output)

tensor([[-0.1363],
        [-0.1364],
        [-0.1315],
        [-0.1184],
        [-0.1256]], grad_fn=<AddmmBackward0>)
